## Importig The Necessary Library

In [ ]:
import numpy as np
import pandas as pd
import torch
import itertools
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.preprocessing import LabelBinarizer
from sklearn.preprocessing import OneHotEncoder
from pymoo.core.problem import Problem
from pymoo.core.problem import ElementwiseProblem
from pymoo.optimize import minimize
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.operators.crossover.sbx import SimulatedBinaryCrossover
from pymoo.operators.mutation.pm import PolynomialMutation
from pymoo.config import Config
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from keras.models import Sequential
from keras.layers import Dense
from sklearn.metrics import confusion_matrix, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score
import shap
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.util.ref_dirs import get_reference_directions
from mpl_toolkits.mplot3d import Axes3D
from sklearn.model_selection import KFold

## Loading The Dataset

In [ ]:
''' Replace '?' with NaN 
    Converting all columns except Labels to numeric
    Computing mean of each numeric column (ignoring NaNs and treating 0 as NaN too)
    Replace NaN values with mean '''

df = pd.read_csv("Immunotherapy.csv")

df.replace('?', np.nan, inplace=True)


for col in df.columns:
    if col != "Class":
        df[col] = pd.to_numeric(df[col], errors='coerce')

means = df.drop(columns=["Class"]).replace(0, np.nan).mean()


df.fillna(means, inplace=True)

X = df.iloc[:, 0:7].values
scaler = MinMaxScaler()
X_imm = scaler.fit_transform(X)

y_imm = df['Class'].map({0: 0, 1: 1}).values

Num_Samples, Num_Features = X_imm.shape
Num_Classes = df['Class'].nunique()
print("Samples:", Num_Samples)
print("Features:", Num_Features)
print("Classes:", Num_Classes)

classes, counts = np.unique(y_imm, return_counts=True)
for cls, cnt in zip(classes, counts):
    print(f"Class {cls}: {cnt} samples")

## Determining The Basic Probability Assignment

In [ ]:
feature_columns = df.columns[df.columns != 'Class']
fig, axes = plt.subplots(nrows = (Num_Features // 4) + 1 , ncols = 4, figsize = (20, 15))

Feature_Density = []
i = 0
k = 0

Array = np.empty((Num_Samples * Num_Features , Num_Classes))

for ax, feature in zip(axes.flatten(), feature_columns):
        plot = sns.histplot(data = df, x = feature, hue = 'Class' , kde = True , ax = ax)
        ax.set_title(f"Distribution of {feature}")
        Feature_1 = df.iloc[: , i]
        Feature_Array = np.zeros((Num_Samples , Num_Classes))
        k = 0
    
        ''' Iterating over each individual feaure value and coputing denity for each class label.'''
    
        for Feature_value in Feature_1:
            intersection_points = []
           
            for cla in df['Class'].unique():
                
                data_points = df[df['Class'] == cla][feature].values
                
                data_points = data_points[~np.isnan(data_points)]

                data_points = np.asarray(data_points, dtype=float)
                num_data_points = len(data_points)
                
                kde_density = gaussian_kde(data_points)(Feature_value) * num_data_points
                intersection_points.append((kde_density)) 
                intersection_array = np.array(intersection_points)
                
            Feature_Array[k , :] = intersection_array.flatten()
            k = k+1
        i = i+1
        Feature_Density.append(Feature_Array)
    
plt.tight_layout()
plt.show()

## Computing The Final Evidential Belief Fetures using Choquet Integral

In [ ]:
Generated_Features = np.zeros((Num_Samples , Num_Classes))
Num_Row , Num_Column = Feature_Density[0].shape
print(Num_Row , Num_Column)

def min_max_normalize(array):

    col_min = np.min(array, axis=0)
    col_max = np.max(array, axis=0)
    
    normalized_array = (array - col_min) / (col_max - col_min)
    
    return normalized_array
    
Normalized_Feature_Density = [min_max_normalize(arr) for arr in Feature_Density]

''' Selecting the q value for the power measure in Choquet integral to aggregate the obtaine beleif degree (BPA) in previous step '''
q = 0.7

for i in range(Num_Row):
    for j in range(Num_Column):
        values = []
        for l in range(len(Normalized_Feature_Density)):
            values.append(Normalized_Feature_Density[l][i][j])
        Sorted_Values = sorted(values)
        Sum = 0
        for k in range(len(Sorted_Values)):
            if (k-1 == -1):
                Sum = Sum + (Sorted_Values[k]*((len(Sorted_Values)-k) / len(Sorted_Values))**q)
            else:
                Sum = Sum + (Sorted_Values[k] - Sorted_Values[k-1])*((len(Sorted_Values)-k) / len(Sorted_Values))**q

        Generated_Features[i][j] += Sum

print(Generated_Features.shape)

''' Normalizing to satisy BPA condition. '''

def normalize_columnwise_sum(array):
    col_sums = np.sum(array, axis=0)
    normalized_array = array / col_sums
    
    return normalized_array

Normalized_Generated_Features = normalize_columnwise_sum(Generated_Features)

## Scaling Evidential Features and Comined with Predefined Features followed by converting the labels into One Hot Encoder

In [ ]:
Normalized_Generated_Features = scaler.fit_transform(Normalized_Generated_Features)
X_Evidential = np.concatenate((X_imm , Normalized_Generated_Features) , axis = 1)
print(X_Evidential.shape)

one_hot_encoder = LabelBinarizer()
Y = one_hot_encoder.fit_transform(y_imm)
print(Y[:5])
print(y_imm[:5])

## Converting the Data into tensor

In [ ]:
# # Convert to torch tensors
device = torch.device("cpu")
X_tensor = torch.tensor(X_Evidential, dtype=torch.float32)
Y_tensor = torch.tensor(Y, dtype=torch.long)

X_tensor = X_tensor.to(device)
Y_tensor = Y_tensor.to(device)

# Print shapes
print(X_tensor.shape)  
print(Y_tensor.shape)  

## Defining A Three Layered Neural Network (NN) Model

In [ ]:
class NN_M(nn.Module):
    def __init__(self , input_size , hidden_size , output_size):
        super(NN_M , self).__init__()
        self.fc1 = nn.Linear(input_size , hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size , output_size)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out

## Defining the Problem with Inclusion of Explaianbility in training Procedure of NN

In [ ]:
class NN_M_NSGA(ElementwiseProblem):
    
    def __init__(self, data, labels, input_size, hidden_size, output_size, feature_index, device=device):
        
        super().__init__(n_var=(input_size * hidden_size) + hidden_size + (hidden_size * output_size) + output_size, 
                         n_obj=3,  
                         xl=-50, 
                         xu=50)
        
        data = data.to(device)
        labels = labels.to(device)
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.feature_index = feature_index  
        self.device = device
        self.data = data.to(self.device)
        self.labels = labels.to(self.device)
        
    def _evaluate(self, x, out, *args, **kwargs):
        
        idx = 0
        
        '''Extracting weights and biases for the first layer'''
        
        weights1 = x[idx:idx + self.input_size * self.hidden_size].reshape(self.hidden_size, self.input_size)
        idx += self.input_size * self.hidden_size
        bias1 = x[idx:idx + self.hidden_size]
        idx += self.hidden_size
        
        ''' Extracting weights and biases for the second layer'''
        
        weights2 = x[idx:idx + self.hidden_size * self.output_size].reshape(self.output_size, self.hidden_size)
        idx += self.hidden_size * self.output_size
        bias2 = x[idx:idx + self.output_size]
        
        ''' Initializing the model and set weights'''
        
        model = NN_M(self.input_size, self.hidden_size, self.output_size)
        model = model.to(self.device)
        model.fc1.weight.data = torch.tensor(weights1).float()
        model.fc1.bias.data = torch.tensor(bias1).float()
        model.fc2.weight.data = torch.tensor(weights2).float()
        model.fc2.bias.data = torch.tensor(bias2).float()
        
        '''Defining the loss function'''
        
        criterion = nn.BCELoss()
        
        
        model = model.to(device)

        self.data = self.data.to(self.device)
        self.labels = self.labels.to(self.device)
            
        outputs = model(self.data)
        loss = criterion(outputs, self.labels.float())

        ''' Using torch.no_grad for consistency'''
            
        with torch.no_grad():      
            outputs = model(self.data)  
            loss = criterion(outputs, self.labels.float()).item()
        
            
        explainer = shap.DeepExplainer(model, self.data)
        shap_values = explainer.shap_values(self.data , check_additivity=False)
        
        if isinstance(shap_values, list):
            shap_values = np.mean(np.stack(shap_values), axis=0)
        shap_values = np.array(shap_values)
        
        if (np.sum(shap_values[shap_values>0]) != 0):
            postive_impact = np.mean(shap_values[shap_values>0])
        else:
            postive_impact = 0
        if (np.sum(shap_values[shap_values<0]) != 0):
            negative_impact = np.mean(shap_values[shap_values>0])
        else:
            negative_impact = 0

        ''' Inclusion of explainability as an objective functions and optimized the model using the below objective space'''
        
        out["F"] = [loss, -postive_impact, negative_impact]   

## Evaluatig The Proposed Model Perfomance

In [ ]:
def Best_Eval(res, X_val, y_val, device=device):
    Acc = 0.0
    X_val = X_val.to(device)
    y_val = y_val.to(device)
    for l in range(len(res.F)):
        best_solution = res.X[l]
        
        idx = 0
        best_weights1 = best_solution[idx:idx + input_size * hidden_size].reshape(hidden_size, input_size)
        idx += input_size * hidden_size
        best_bias1 = best_solution[idx:idx + hidden_size]
        idx += hidden_size
        best_weights2 = best_solution[idx:idx + hidden_size * output_size].reshape(output_size, hidden_size)
        idx += hidden_size * output_size
        best_bias2 = best_solution[idx:idx + output_size]
    
        ''' Creating the optimized model '''
        
        optimized_model = NN_M(input_size, hidden_size, output_size).to(device)
        optimized_model.fc1.weight.data = torch.tensor(best_weights1).float()
        optimized_model.fc1.bias.data = torch.tensor(best_bias1).float()
        optimized_model.fc2.weight.data = torch.tensor(best_weights2).float()
        optimized_model.fc2.bias.data = torch.tensor(best_bias2).float()
    
        ''' Evaluating the optimized model '''
        
        optimized_model.eval()
        
        with torch.no_grad():
            outputs = optimized_model(X_val)
            predictions = (outputs > 0.5).float().numpy()
            labels = y_val.numpy()

            tp = np.sum((predictions == 1) & (labels == 1))
            fn = np.sum((predictions == 0) & (labels == 1))
            tn = np.sum((predictions == 0) & (labels == 0))
            fp = np.sum((predictions == 1) & (labels == 0))

            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            tp_tn = tp + tn
            accuracy = accuracy_score(labels, predictions)
            auc = roc_auc_score(labels, predictions)
            precision = precision_score(labels, predictions)
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
            f1 = f1_score(labels, predictions)
            kappa = cohen_kappa_score(labels, predictions)

            if (Acc<accuracy):
                Acc = accuracy
                k=l
    return k

In [ ]:
def Eval(res, X_val, y_val, l, device=device):
    
        best_solution = res.X[l]
        X_val = X_val.to(device)
        y_val = y_val.to(device)
        best_solution = res.X[l]
    
        idx = 0
        best_weights1 = best_solution[idx:idx + input_size * hidden_size].reshape(hidden_size, input_size)
        idx += input_size * hidden_size
        best_bias1 = best_solution[idx:idx + hidden_size]
        idx += hidden_size
        best_weights2 = best_solution[idx:idx + hidden_size * output_size].reshape(output_size, hidden_size)
        idx += hidden_size * output_size
        best_bias2 = best_solution[idx:idx + output_size]
    
        ''' Creating the optimized model '''
        optimized_model = NN_M(input_size, hidden_size, output_size).to(device)
        optimized_model.fc1.weight.data = torch.tensor(best_weights1).float()
        optimized_model.fc1.bias.data = torch.tensor(best_bias1).float()
        optimized_model.fc2.weight.data = torch.tensor(best_weights2).float()
        optimized_model.fc2.bias.data = torch.tensor(best_bias2).float()
    
        ''' Evaluating the optimized model '''
    
        optimized_model.eval()
    
        with torch.no_grad():
            outputs = optimized_model(X_val)
            predictions = (outputs > 0.5).float().numpy()
            labels = y_val.numpy()

            tp = np.sum((predictions == 1) & (labels == 1))
            fn = np.sum((predictions == 0) & (labels == 1))
            tn = np.sum((predictions == 0) & (labels == 0))
            fp = np.sum((predictions == 1) & (labels == 0))

            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            tp_tn = tp + tn
            accuracy = accuracy_score(labels, predictions)
            auc = roc_auc_score(labels, predictions)
            precision = precision_score(labels, predictions)
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
            f1 = f1_score(labels, predictions)
            kappa = cohen_kappa_score(labels, predictions)

        results = {
            'AUC': roc_auc_score(labels, outputs),  
            'Accuracy': accuracy,
            'Sensitivity': sensitivity,
            'Precision': precision,
            'Specificity': specificity,
            'F1 Score': f1,
            'Kappa': kappa
        }
        return results

## Initializing The HyperParameters and Optimizing the Model using NSGA III

In [ ]:
hidden_size = 16
n_obj = 3
all_results = []
input_size = X_Evidential.shape[1]
output_size = 1  
feature_index = input_size - 1  
Results = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_tensor, Y_tensor)):##(X_Evidential, y_imm)):
    X_train, X_val = X_tensor[train_idx], X_tensor[val_idx]
    y_train, y_val = Y_tensor[train_idx], Y_tensor[val_idx]

    X_train, X_val = X_train.to(device), X_val.to(device)
    y_train, y_val = y_train.to(device), y_val.to(device)
    
    problem = NN_M_NSGA(X_train, y_train, input_size, hidden_size, output_size, feature_index, device=device)
    
    Config.warnings['not_compiled'] = False
    ref_dirs = get_reference_directions("energy", n_obj, n_points=10)
    pop_size = len(ref_dirs)
    
    ''' Defining the algorithm '''
    
    algorithm = NSGA3(
        ref_dirs=ref_dirs,
        pop_size=pop_size,
        sampling=FloatRandomSampling(),
        crossover=SimulatedBinaryCrossover(prob=0.9, eta=15),
        mutation=PolynomialMutation(eta=20),
        eliminate_duplicates=True
    )
    
    ''' Optimizing the defined problem '''
    
    res = minimize(problem,
                   algorithm,
                   termination = ('n_gen', 15),
                   seed = 1,
                   verbose = True,
                  )

    l = Best_Eval(res, X_val, y_val, device=device)
    Results.append(Eval(res, X_val, y_val, l, device=device))

## Computing Mean and Std for the Perfomance Metrics

In [ ]:
keys = Results[0].keys()

mean_results = {}
std_results = {}

for k in keys:
    values = [r[k] for r in Results]   
    mean_results[k] = np.mean(values)
    std_results[k] = np.std(values, ddof=1)  

print("Mean:", mean_results)
print("Std :", std_results)
